# Vehicle Classification – 03 · Models

Sections 6–13 of the report:

1. Validation strategy
2. Basic models – logistic regression, decision tree, Naive Bayes
3. Advanced models – discriminant analysis, SVM, bagging, random forests, boosting (XGBoost and
   CatBoost on the GPU), neural networks (batched MLPs and an FT-Transformer on the GPU), TabPFN
4. Ensembles, stacking and threshold tuning
5. Robustness to measurement noise
6. Fresh-fold confirmation
7. Final model and submission file
8. Experiments with 20% of the training data
9. Unsupervised learning
10. Key insights

Data handling, the test-noise model and the validation protocol are explained in
`02_Preprocessing`; the setup cell below repeats them so this notebook runs on its own.
GPU-trained models (MLP, FT-Transformer, XGBoost, CatBoost, TabPFN) can differ in the second
decimal between runs.

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as Fn
from joblib import Parallel, delayed
from scipy.optimize import linear_sum_assignment
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
from sklearn.ensemble import (AdaBoostClassifier, BaggingClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier, HistGradientBoostingClassifier,
                              RandomForestClassifier)
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import (adjusted_rand_score, completeness_score, confusion_matrix,
                             homogeneity_score, precision_recall_fscore_support, silhouette_score)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import StratifiedKFold
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, PowerTransformer, SplineTransformer, StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = True
print("device:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu")

CLASS = {0: ("Sedan", "#2a78d6"), 1: ("SUV", "#eb6834"), 2: ("Sports Car", "#1baf7a")}
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.edgecolor": AXIS, "axes.linewidth": 0.6,
    "axes.labelcolor": INK2, "axes.titlesize": 10, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "xtick.major.size": 0, "ytick.major.size": 0, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.spines.top": False,
    "axes.spines.right": False, "legend.frameon": False})

In [ ]:
# ---- data -----------------------------------------------------------------------------
TARGET, ID = "vehicle_class", "vehicle_id"
train_raw = pd.read_csv("data/vehicle_train_10000.csv")
test_raw = pd.read_csv("data/vehicle_test.csv")
FEATURES = [c for c in train_raw.columns if c not in (ID, TARGET)]
train = train_raw.drop_duplicates(subset=FEATURES + [TARGET]).reset_index(drop=True)
X, y = train[FEATURES].to_numpy(float), train[TARGET].to_numpy(int)
X_test, test_ids = test_raw[FEATURES].to_numpy(float), test_raw[ID].to_numpy()
LO, HI = X.min(0), X.max(0)                                          # clip limits
DEC = np.array([0 if np.all(np.round(c) == c) else 2 for c in X.T])  # recorded precision
print(f"{len(X):,} training vehicles (after de-duplication), {len(X_test):,} test vehicles, "
      f"{len(FEATURES)} features")


# ---- the measurement noise of the test file ---------------------------------------------
def test_noise(Xv, s, share, col_sd, rng):
    """N(0, s * feature sd) on a random `share` of the values, then re-clipped to the
    limits and re-rounded to the recorded precision - the pattern of the test file."""
    if s <= 0:
        return Xv.copy()
    E = rng.normal(0.0, 1.0, Xv.shape) * (s * col_sd)
    if share < 1:
        E *= rng.random(Xv.shape) < share
    Z = np.clip(Xv + E, LO, HI)
    for j, d in enumerate(DEC):
        Z[:, j] = np.round(Z[:, j], d)
    return Z


# ---- training-time augmentation -----------------------------------------------------------
def jitter_copies(Xa, ya, sd=0.20, copies=4, seed=42):
    """Training rows + `copies` Gaussian-jittered copies (the final model's augmentation)."""
    rng = np.random.default_rng(seed)
    col_sd = Xa.std(0, ddof=1)
    parts = [Xa] + [Xa + rng.normal(0.0, sd * col_sd, Xa.shape) for _ in range(copies)]
    return np.vstack(parts), np.concatenate([ya] * (copies + 1))


def test_noise_copies(Xa, ya, s=0.15, share=1.0, copies=4, seed=0):
    """Training rows + copies carrying the test-file noise exactly (clip + round)."""
    rng = np.random.default_rng(seed)
    col_sd = Xa.std(0, ddof=1)
    parts = [Xa] + [test_noise(Xa, s, share, col_sd, rng) for _ in range(copies)]
    return np.vstack(parts), np.tile(ya, copies + 1)


def alternating_noise_copies(Xa, ya, K, seed=0):
    """K test-noise copies alternating the two calibrated readings (0.08 sd / 0.12 sd on half)."""
    rng = np.random.default_rng(seed)
    col_sd = Xa.std(0, ddof=1)
    parts = [Xa] + [test_noise(Xa, *((0.08, 1.0) if k % 2 == 0 else (0.12, 0.5)), col_sd, rng)
                    for k in range(K)]
    return np.vstack(parts), np.tile(ya, K + 1)


# ---- validation protocol --------------------------------------------------------------
VARIANTS = {            # name: (noise sd as a fraction of feature sd, share of values hit, draws)
    "clean": (0.00, 1.0, 1),
    "u08":   (0.08, 1.0, 2),     # reading 1 of the test noise: 0.08 sd on every value
    "mix12": (0.12, 0.5, 2),     # reading 2: 0.12 sd on a random half of the values
    "u15":   (0.15, 1.0, 1),     # stress level
}
SCREEN_SEEDS, CONFIRM_SEEDS = (42, 7, 123), (2024, 99, 314)


def make_folds(seeds):
    """Repeated stratified 5-fold CV; every validation fold carries its noisy copies."""
    folds = []
    for seed in seeds:
        for k, (tr, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y)):
            sd = X[tr].std(0, ddof=1)
            variants = {name: [test_noise(X[va], s, share, sd, np.random.default_rng([seed, k, vi, r]))
                               for r in range(reps)]
                        for vi, (name, (s, share, reps)) in enumerate(VARIANTS.items())}
            folds.append({"key": f"{seed}_{k}", "tr": tr, "va": va, "variants": variants})
    return folds


def proba(model, Z):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(Z)
    d = model.decision_function(Z)                      # margin models: softmax of the scores
    e = np.exp(d - d.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


P = {}      # model name -> one entry per fold: {variant: [probabilities for each noise draw]}


def run(name, fit, folds=None, store=None, n_jobs=-1):
    """fit(X_train, y_train) -> fitted model. Scores every noisy copy of every validation fold."""
    folds = FOLDS if folds is None else folds
    store = P if store is None else store

    def one(f):
        m = fit(X[f["tr"]], y[f["tr"]])
        return {v: [proba(m, Z) for Z in Zs] for v, Zs in f["variants"].items()}

    t = time.time()
    store[name] = Parallel(n_jobs=n_jobs)(delayed(one)(f) for f in folds)
    print(f"{name:<52} {time.time() - t:6.0f}s")


def fold_acc(name, folds=None, store=None):
    folds = FOLDS if folds is None else folds
    store = P if store is None else store
    out = []
    for f, probs in zip(folds, store[name]):
        yv = y[f["va"]]
        out.append({v: np.mean([(p.argmax(1) == yv).mean() for p in ps]) for v, ps in probs.items()})
    return out


BASE = "Logistic regression (final)"


def table(names, folds=None, store=None, base=BASE):
    """Accuracy (%) per noise condition and the paired gap to the final model (+- one s.e.)."""
    b = np.array([(a["u08"] + a["mix12"]) / 2 for a in fold_acc(base, folds, store)])
    rows = []
    for n in names:
        acc = fold_acc(n, folds, store)
        tl = np.array([(a["u08"] + a["mix12"]) / 2 for a in acc])
        d = (tl - b[:len(tl)]) * 100
        rows.append({"model": n, "clean": np.mean([a["clean"] for a in acc]) * 100,
                     "test-like": tl.mean() * 100, "stress 0.15": np.mean([a["u15"] for a in acc]) * 100,
                     "gap vs final": d.mean(), "+- s.e.": d.std(ddof=1) / np.sqrt(len(d))})
    return pd.DataFrame(rows).set_index("model").round(2) + 0.0      # + 0.0 turns -0.00 into 0.00


def average(names, store=None):
    """Probability average of stored models (same folds, same noisy copies)."""
    store = P if store is None else store
    first = store[names[0]]
    return [{v: [np.mean([store[n][i][v][r] for n in names], axis=0) for r in range(len(ps))]
             for v, ps in first[i].items()} for i in range(len(first))]


def stacked(f):
    return np.vstack([Z for Zs in f["variants"].values() for Z in Zs])


def unstacked(Pall, f):
    out, i, n = {}, 0, len(f["va"])
    for v, Zs in f["variants"].items():
        out[v] = []
        for _ in Zs:
            out[v].append(Pall[i:i + n])
            i += n
    return out


def logreg(C=0.1, **kw):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=20000, **kw))


class WithLDA(BaseEstimator, ClassifierMixin):
    """Appends the two LDA discriminant scores (fitted on the training rows only), so trees
    can split along the tilted class boundary instead of building a staircase."""

    def __init__(self, base):
        self.base = base

    def _tf(self, Z):
        return np.hstack([Z, self.lda_.transform(self.sc_.transform(Z))])

    def fit(self, Z, t):
        self.sc_ = StandardScaler().fit(Z)
        self.lda_ = LDA(n_components=2).fit(self.sc_.transform(Z), t)
        self.base.fit(self._tf(Z), t)
        self.classes_ = np.unique(t)
        return self

    def predict_proba(self, Z):
        return self.base.predict_proba(self._tf(Z))

    def predict(self, Z):
        return self.predict_proba(Z).argmax(1)

## 1. Validation strategy

15 screening folds (3 repeats of stratified 5-fold CV) and 15 fresh confirmation folds. Every
validation fold carries the clean rows and noisy copies with the measured test noise; models
are compared on **test-like accuracy** (mean of the two noise readings) with paired standard
errors against the final model.

In [ ]:
FOLDS, CONFIRM = make_folds(SCREEN_SEEDS), make_folds(CONFIRM_SEEDS)
jitter = lambda Xt, yt: jitter_copies(Xt, yt)
print(f"{len(FOLDS)} screening folds, {len(CONFIRM)} confirmation folds")

## 2. Basic models

### 2.1 Logistic regression – baseline → tuned

The baseline is a scaled multinomial logistic regression with C = 1. Tuning sweeps the inverse
regularisation strength C and adds noise-aware training (four jittered copies). The softmax
model is also compared with one-vs-rest, a hinge loss (linear SVM) and least squares (ridge).

In [ ]:
run("Logistic regression - baseline (C = 1)", lambda Xt, yt: logreg(1.0).fit(Xt, yt))
run(BASE, lambda Xt, yt: logreg(0.1).fit(*jitter(Xt, yt)))
for C in (0.01, 0.03, 0.3, 1.0):
    run(f"Logistic regression C = {C} + augmentation", lambda Xt, yt, C=C: logreg(C).fit(*jitter(Xt, yt)))
run("Logistic regression one-vs-rest", lambda Xt, yt: make_pipeline(
    StandardScaler(), OneVsRestClassifier(LogisticRegression(C=0.1, max_iter=20000))).fit(*jitter(Xt, yt)))
run("Linear SVM (hinge loss)", lambda Xt, yt: make_pipeline(
    StandardScaler(), LinearSVC(C=0.05, dual=False)).fit(*jitter(Xt, yt)))
run("Ridge classifier (least squares)", lambda Xt, yt: make_pipeline(
    StandardScaler(), RidgeClassifier(alpha=10.0)).fit(*jitter(Xt, yt)))
table([n for n in P if "ogistic" in n or "SVM" in n or "Ridge" in n])

Accuracy is flat from C = 0.01 to 1, and **C = 0.1 with noise augmentation** is the best model of
the project. The class boundary is essentially linear, and the L2 penalty stops the model from
leaning on the tiny, noise-sensitive directions of an ill-conditioned covariance.

### 2.2 Decision tree

In [ ]:
tree = lambda **k: DecisionTreeClassifier(random_state=0, **k)
run("Decision tree - depth 6", lambda Xt, yt: tree(max_depth=6, min_samples_leaf=20).fit(Xt, yt))
run("Decision tree - depth 10", lambda Xt, yt: tree(max_depth=10, min_samples_leaf=10).fit(Xt, yt))
run("Decision tree - depth 4 + LDA features",
    lambda Xt, yt: WithLDA(tree(max_depth=4, min_samples_leaf=20)).fit(Xt, yt))
table([n for n in P if n.startswith("Decision tree")])

A tree splits on one feature at a time, so it approximates the tilted linear boundary with a
staircase. Giving it the two LDA discriminant directions as extra features helps a lot, but it
stays well below the linear model.

### 2.3 Naive Bayes

Gaussian Naive Bayes assumes the features are independent given the class – badly violated
by correlations near 0.95. Rotating to a better coordinate system fixes it.

In [ ]:
run("Gaussian Naive Bayes (raw features)", lambda Xt, yt: GaussianNB().fit(Xt, yt))
run("Naive Bayes + PCA whitening (10 comps)", lambda Xt, yt: make_pipeline(
    StandardScaler(), PCA(10, whiten=True, random_state=42), GaussianNB()).fit(*jitter(Xt, yt)))
run("Naive Bayes + LDA(2) projection + aug.", lambda Xt, yt: make_pipeline(
    StandardScaler(), LDA(n_components=2), GaussianNB()).fit(*jitter(Xt, yt)))
basic = table(["Logistic regression - baseline (C = 1)", BASE, "Decision tree - depth 6",
               "Decision tree - depth 10", "Decision tree - depth 4 + LDA features",
               "Gaussian Naive Bayes (raw features)", "Naive Bayes + PCA whitening (10 comps)",
               "Naive Bayes + LDA(2) projection + aug."])
basic

## 3. Advanced models

### 3.1 Discriminant analysis (Bayes with full covariance)

In [ ]:
class RDA(BaseEstimator, ClassifierMixin):
    """Friedman's regularised discriminant analysis: class covariances blended with the pooled
    one (alpha: 0 = LDA, 1 = QDA) and shrunk toward a scaled identity (gamma)."""
    def __init__(self, alpha=0.1, gamma=0.02):
        self.alpha, self.gamma = alpha, gamma
    def fit(self, Xa, ya):
        self.sc_ = StandardScaler().fit(Xa)
        Z = self.sc_.transform(Xa)
        p = Z.shape[1]
        self.mu_ = np.stack([Z[ya == k].mean(0) for k in range(3)])
        covs = [np.cov(Z[ya == k], rowvar=False) for k in range(3)]
        pooled = sum((np.sum(ya == k) - 1) * covs[k] for k in range(3)) / (len(ya) - 3)
        self.inv_, self.logdet_ = [], []
        for k in range(3):
            S = self.alpha * covs[k] + (1 - self.alpha) * pooled
            S = (1 - self.gamma) * S + self.gamma * np.trace(S) / p * np.eye(p)
            self.inv_.append(np.linalg.inv(S))
            self.logdet_.append(np.linalg.slogdet(S)[1])
        self.logprior_ = np.log(np.bincount(ya) / len(ya))
        self.classes_ = np.arange(3)
        return self
    def predict_proba(self, Za):
        Z = self.sc_.transform(Za)
        L = np.column_stack([-0.5 * np.einsum("ij,jk,ik->i", Z - self.mu_[k], self.inv_[k], Z - self.mu_[k])
                             - 0.5 * self.logdet_[k] + self.logprior_[k] for k in range(3)])
        Pk = np.exp(L - L.max(1, keepdims=True))
        return Pk / Pk.sum(1, keepdims=True)

class MixtureDA(BaseEstimator, ClassifierMixin):
    """Mixture discriminant analysis: a Gaussian mixture per class."""
    def __init__(self, k=4, cov="tied"):
        self.k, self.cov = k, cov
    def fit(self, Xa, ya):
        self.sc_ = StandardScaler().fit(Xa)
        Z = self.sc_.transform(Xa)
        self.prior_ = np.bincount(ya) / len(ya)
        self.gm_ = [GaussianMixture(self.k, covariance_type=self.cov, reg_covar=1e-2, n_init=3,
                                    random_state=0).fit(Z[ya == c]) for c in range(3)]
        self.classes_ = np.arange(3)
        return self
    def predict_proba(self, Za):
        Z = self.sc_.transform(Za)
        L = np.column_stack([g.score_samples(Z) for g in self.gm_]) + np.log(self.prior_)
        Pk = np.exp(L - L.max(1, keepdims=True))
        return Pk / Pk.sum(1, keepdims=True)

run("LDA, covariance shrinkage 0.1", lambda Xt, yt: make_pipeline(
    StandardScaler(), LDA(solver="lsqr", shrinkage=0.1)).fit(Xt, yt))
run("QDA, reg_param 0.1", lambda Xt, yt: make_pipeline(StandardScaler(), QDA(reg_param=0.1)).fit(Xt, yt))
run("Regularised DA (alpha = 0.1)", lambda Xt, yt: RDA(0.1, 0.02).fit(Xt, yt))
run("Mixture DA (4 tied Gaussians / class)", lambda Xt, yt: MixtureDA(4, "tied").fit(Xt, yt))
table(["LDA, covariance shrinkage 0.1", "QDA, reg_param 0.1", "Regularised DA (alpha = 0.1)",
       "Mixture DA (4 tied Gaussians / class)"])

LDA (shared covariance, linear boundary) ties logistic regression; QDA and the mixture models,
which allow class-specific shapes, score lower. The classes share essentially one covariance –
exactly the case where the linear boundary is optimal.

### 3.2 SVM

In [ ]:
run("SVM, RBF kernel, C = 1", lambda Xt, yt: make_pipeline(
    StandardScaler(), SVC(C=1.0, probability=True, random_state=0)).fit(Xt, yt))
table(["SVM, RBF kernel, C = 1", "Linear SVM (hinge loss)"])

### 3.3 Bagging and random forests

In [ ]:
rf = lambda **k: RandomForestClassifier(n_estimators=500, n_jobs=1, random_state=0, **k)
run("Bagged decision trees (300)", lambda Xt, yt: BaggingClassifier(
    tree(), n_estimators=300, max_samples=0.8, n_jobs=1, random_state=0).fit(Xt, yt))
run("Bagged logistic regression (50)", lambda Xt, yt: BaggingClassifier(
    make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000)),
    n_estimators=50, max_samples=0.8, n_jobs=1, random_state=0).fit(Xt, yt))
run("Random forest (500 trees)", lambda Xt, yt: rf().fit(Xt, yt))
run("Random forest + LDA features + noise aug.",
    lambda Xt, yt: WithLDA(rf(min_samples_leaf=5)).fit(*test_noise_copies(Xt, yt, 0.15)))
run("Extra-trees + LDA features", lambda Xt, yt: WithLDA(ExtraTreesClassifier(
    n_estimators=500, min_samples_leaf=5, n_jobs=1, random_state=0)).fit(Xt, yt))
table(["Bagged decision trees (300)", "Bagged logistic regression (50)", "Random forest (500 trees)",
       "Random forest + LDA features + noise aug.", "Extra-trees + LDA features"])

Averaging trees smooths the staircase boundary, and the LDA directions help most – but even the
best forest stays clearly below the linear model. Bagging a stable model (logistic regression)
changes almost nothing.

### 3.4 Boosting (XGBoost and CatBoost on the GPU)

In [ ]:
xgb = lambda depth, n, rate: XGBClassifier(device="cuda", tree_method="hist", n_estimators=n, max_depth=depth,
                                           learning_rate=rate, subsample=0.8, colsample_bytree=0.8,
                                           min_child_weight=5, verbosity=0)
run("AdaBoost (depth-2 trees)", lambda Xt, yt: AdaBoostClassifier(
    tree(max_depth=2), n_estimators=300, learning_rate=0.3, random_state=0).fit(Xt, yt))
run("Gradient boosting (scikit-learn)", lambda Xt, yt: GradientBoostingClassifier(
    n_estimators=300, max_depth=2, learning_rate=0.05, subsample=0.8, random_state=0).fit(Xt, yt))
run("HistGradientBoosting", lambda Xt, yt: HistGradientBoostingClassifier(
    max_depth=3, learning_rate=0.05, max_iter=500, l2_regularization=1.0, random_state=0).fit(Xt, yt))
run("XGBoost on GPU", lambda Xt, yt: xgb(2, 1200, 0.05).fit(Xt, yt), n_jobs=1)
run("XGBoost on GPU + LDA features", lambda Xt, yt: WithLDA(xgb(2, 600, 0.03)).fit(Xt, yt), n_jobs=1)
run("LightGBM + LDA features + noise aug.", lambda Xt, yt: WithLDA(LGBMClassifier(
    n_estimators=600, num_leaves=7, learning_rate=0.03, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8, min_child_samples=20, verbose=-1, n_jobs=8)).fit(*test_noise_copies(Xt, yt, 0.15)),
    n_jobs=1)
run("CatBoost on GPU + LDA features", lambda Xt, yt: WithLDA(CatBoostClassifier(
    task_type="GPU", iterations=1000, depth=4, learning_rate=0.03, l2_leaf_reg=3, verbose=0,
    random_seed=0, allow_writing_files=False)).fit(Xt, yt), n_jobs=1)
table(["AdaBoost (depth-2 trees)", "Gradient boosting (scikit-learn)", "HistGradientBoosting",
       "XGBoost on GPU", "XGBoost on GPU + LDA features", "LightGBM + LDA features + noise aug.",
       "CatBoost on GPU + LDA features"])

### 3.5 Neural networks on the GPU

**Batched MLPs.** Instead of training one network at a time, hundreds of independent networks
are stacked into batched matrix multiplications (`torch.baddbmm`): every network has its own
weights, fold, input-noise level, weight decay and dropout. Here 4 configurations × 15 folds × 3
seeds = 180 networks train together; the report's full grid was 54 configurations (2,430 networks).

In [ ]:
class BatchedMLP(nn.Module):
    """M independent MLPs evaluated together: x has shape (M, batch, features)."""
    def __init__(self, M, d_in, width, depth, n_out=3, seed=0):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        dims = [d_in] + [width] * depth + [n_out]
        self.W, self.b = nn.ParameterList(), nn.ParameterList()
        for a, b in zip(dims[:-1], dims[1:]):
            bound = np.sqrt(6.0 / a) if b != n_out else np.sqrt(1.0 / a)
            self.W.append(nn.Parameter((torch.rand(M, a, b, generator=g) * 2 - 1) * bound))
            self.b.append(nn.Parameter(torch.zeros(M, 1, b)))
    def forward(self, x, keep=None):
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            x = torch.baddbmm(b, x, W)
            if i < len(self.W) - 1:
                x = Fn.silu(x)
                if keep is not None:
                    x = x * (torch.rand_like(x) < keep).float() / keep
        return x

def train_mlps(train_sets, eval_sets, width, depth, settings, n_seeds=3, epochs=80, batch=256, lr_max=3e-3):
    """Train len(settings) x len(train_sets) x n_seeds networks in one pass; each injects fresh
    test-like noise into every minibatch. Returns {(setting, set): seed-averaged probabilities}."""
    combos = [(si, ti, s) for si in range(len(settings)) for ti in range(len(train_sets)) for s in range(n_seeds)]
    M = len(combos)
    t_idx = torch.tensor([c[1] for c in combos], device=DEVICE)
    mu = np.stack([Xt.mean(0) for Xt, _ in train_sets])
    sd = np.stack([Xt.std(0) for Xt, _ in train_sets])
    n_max = max(len(Xt) for Xt, _ in train_sets)
    pad = lambda A: np.concatenate([A, A[:n_max - len(A)]]) if len(A) < n_max else A
    XS = torch.tensor(np.stack([pad((Xt - mu[i]) / sd[i]) for i, (Xt, _) in enumerate(train_sets)]),
                      dtype=torch.float32, device=DEVICE)[t_idx]
    YS = torch.tensor(np.stack([pad(yt) for _, yt in train_sets]), device=DEVICE)[t_idx]
    n_tr = torch.tensor([len(train_sets[c[1]][0]) for c in combos], device=DEVICE)[:, None]
    lo = torch.tensor((LO - mu) / sd, dtype=torch.float32, device=DEVICE)[t_idx][:, None]
    hi = torch.tensor((HI - mu) / sd, dtype=torch.float32, device=DEVICE)[t_idx][:, None]
    par = lambda k: torch.tensor([settings[c[0]][k] for c in combos], dtype=torch.float32, device=DEVICE)[:, None, None]
    s_noise, share, wd, keep = par("noise"), par("share"), par("wd"), 1 - par("drop")
    keep = keep if bool((keep < 1).any()) else None
    net = BatchedMLP(M, XS.shape[2], width, depth, seed=1000 * width + depth).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=lr_max)
    steps = epochs * (n_max // batch)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr_max, total_steps=steps, pct_start=0.1)
    gen = torch.Generator(device=DEVICE).manual_seed(7 * width + depth)
    for _ in range(steps):
        idx = (torch.rand(M, batch, device=DEVICE, generator=gen) * n_tr).long()
        xb = torch.gather(XS, 1, idx[..., None].expand(-1, -1, XS.shape[2]))
        yb = torch.gather(YS, 1, idx)
        e = torch.randn(xb.shape, device=DEVICE, generator=gen) * s_noise
        e = e * (torch.rand(xb.shape, device=DEVICE, generator=gen) < share).float()
        xb = torch.minimum(torch.maximum(xb + e, lo), hi)
        loss = Fn.cross_entropy(net(xb, keep).reshape(-1, 3), yb.reshape(-1), reduction="none")
        loss = loss.view(M, batch).mean(1).sum()
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        with torch.no_grad():                               # decoupled, per-network weight decay
            for p in net.W:
                p.mul_(1 - sched.get_last_lr()[0] * wd)
        sched.step()
    n_ev = max(len(Z) for Z in eval_sets)
    pad_ev = lambda A: np.concatenate([A, A[:n_ev - len(A)]]) if len(A) < n_ev else A
    ZE = torch.tensor(np.stack([pad_ev((Z - mu[i]) / sd[i]) for i, Z in enumerate(eval_sets)]),
                      dtype=torch.float32, device=DEVICE)
    Pm = torch.empty(M, n_ev, 3)
    with torch.no_grad():
        for i in range(0, n_ev, 1000):
            Pm[:, i:i + 1000] = torch.softmax(net(ZE[t_idx, i:i + 1000]), -1).cpu()
    Pm = Pm.numpy()
    return {(si, ti): Pm[[m for m, c in enumerate(combos) if c[:2] == (si, ti)], :len(eval_sets[ti])].mean(0)
            for si in range(len(settings)) for ti in range(len(eval_sets))}

MLP_CONFIGS = {   # name: (width, depth, settings)
    "MLP 64x2, input noise 0.2, dropout 0.1":   (64, 2, dict(noise=0.2, share=1.0, wd=0.01, drop=0.1)),
    "MLP 64x2, no regularisation":              (64, 2, dict(noise=0.0, share=1.0, wd=1e-4, drop=0.0)),
    "MLP 256x2, input noise 0.2, dropout 0.1":  (256, 2, dict(noise=0.2, share=1.0, wd=0.01, drop=0.1)),
    "MLP 256x3, input noise 0.3, dropout 0.1":  (256, 3, dict(noise=0.3, share=1.0, wd=0.01, drop=0.1)),
}
for arch in sorted({(w, d) for w, d, _ in MLP_CONFIGS.values()}):
    names = [n for n, (w, d, _) in MLP_CONFIGS.items() if (w, d) == arch]
    t = time.time()
    probs = train_mlps([(X[f["tr"]], y[f["tr"]]) for f in FOLDS], [stacked(f) for f in FOLDS],
                       *arch, [MLP_CONFIGS[n][2] for n in names])
    for si, n in enumerate(names):
        P[n] = [unstacked(probs[(si, i)], f) for i, f in enumerate(FOLDS)]
    print(f"MLP {arch[0]}x{arch[1]}: {len(names) * len(FOLDS) * 3} networks trained together in {time.time() - t:.0f}s")
table(list(MLP_CONFIGS))

The trend is monotonic: **more capacity, lower accuracy**, and every bit of regularisation
helps – pushing the network towards a linear model.

**FT-Transformer** – each feature becomes a token and a [CLS] token attends over them
(trained from scratch, with input-noise injection).

In [ ]:
class FTTransformer(nn.Module):
    def __init__(self, n_feat, d=32, layers=2, heads=8, drop=0.1):
        super().__init__()
        self.w = nn.Parameter(torch.randn(n_feat, d) / d ** 0.5)
        self.b = nn.Parameter(torch.zeros(n_feat, d))
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        layer = nn.TransformerEncoderLayer(d, heads, 2 * d, drop, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.GELU(), nn.Linear(d, 3))
    def forward(self, x):
        tok = torch.cat([self.cls.expand(len(x), -1, -1), x[..., None] * self.w + self.b], 1)
        return self.head(self.enc(tok)[:, 0])

def ft_fit_predict(Xtr, ytr, Z, d, layers, noise, epochs=60, seed=0):
    torch.manual_seed(seed)
    mu, sd = Xtr.mean(0), Xtr.std(0)
    lo = torch.tensor((LO - mu) / sd, dtype=torch.float32, device=DEVICE)
    hi = torch.tensor((HI - mu) / sd, dtype=torch.float32, device=DEVICE)
    XT = torch.tensor((Xtr - mu) / sd, dtype=torch.float32, device=DEVICE)
    YT = torch.tensor(ytr, device=DEVICE)
    net = FTTransformer(X.shape[1], d, layers).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    steps = epochs * (len(XT) // 256)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 1e-3, total_steps=steps, pct_start=0.1)
    net.train()
    for _ in range(steps):
        idx = torch.randint(0, len(XT), (256,), device=DEVICE)
        xb = torch.minimum(torch.maximum(XT[idx] + torch.randn_like(XT[idx]) * noise, lo), hi)
        loss = Fn.cross_entropy(net(xb), YT[idx])
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        sched.step()
    net.eval()
    with torch.no_grad():
        Zt = torch.tensor((Z - mu) / sd, dtype=torch.float32, device=DEVICE)
        return torch.cat([torch.softmax(net(Zt[i:i + 2048]), -1) for i in range(0, len(Zt), 2048)]).cpu().numpy()

for name, (d, layers, noise) in {"FT-Transformer (small: d=32, 2 layers)": (32, 2, 0.20),
                                 "FT-Transformer (large: d=64, 3 layers)": (64, 3, 0.15)}.items():
    t = time.time()
    P[name] = [unstacked(ft_fit_predict(X[f["tr"]], y[f["tr"]], stacked(f), d, layers, noise), f) for f in FOLDS]
    print(f"{name:<52} {time.time() - t:6.0f}s")
table(["FT-Transformer (small: d=32, 2 layers)", "FT-Transformer (large: d=64, 3 layers)"])

### 3.6 TabPFN v2 – a pretrained tabular transformer

TabPFN is pretrained to perform Bayesian inference on small tables. It is slow (about 80 s per
fold on an RTX 4060), so it runs on the 5 folds of the first CV repeat and is compared with
logistic regression on those same folds. The v2 weights are openly licensed (Prior Labs
License 1.1) and download without an account.

In [ ]:
RUN_TABPFN = True
if RUN_TABPFN:
    from tabpfn import TabPFNClassifier
    from tabpfn.constants import ModelVersion
    t = time.time()
    P["TabPFN v2 (5 folds)"] = []
    for f in FOLDS[:5]:
        m = TabPFNClassifier.create_default_for_version(
            ModelVersion.V2, device=DEVICE.type, n_estimators=4, memory_saving_mode=True,
            fit_mode="fit_preprocessors", random_state=0).fit(X[f["tr"]], y[f["tr"]])
        Zs = stacked(f)
        P["TabPFN v2 (5 folds)"].append(unstacked(
            np.vstack([m.predict_proba(Zs[i:i + 1500]) for i in range(0, len(Zs), 1500)]), f))
    print(f"TabPFN: {time.time() - t:.0f}s")
    display(table([BASE, "TabPFN v2 (5 folds)"], folds=FOLDS[:5]))

### Summary: best configuration of each family

In [ ]:
run("Spline GAM + logistic regression", lambda Xt, yt: make_pipeline(
    SplineTransformer(n_knots=5, degree=3, extrapolation="linear"), StandardScaler(),
    LogisticRegression(C=0.05, max_iter=20000)).fit(*jitter(Xt, yt)))
FAMILY_BEST = {
    "Logistic regression (final)": BASE,
    "Naive Bayes + LDA(2)": "Naive Bayes + LDA(2) projection + aug.",
    "LDA (shrinkage)": "LDA, covariance shrinkage 0.1",
    "Spline GAM": "Spline GAM + logistic regression",
    "SVM (RBF)": "SVM, RBF kernel, C = 1",
    "MLP (GPU)": "MLP 64x2, input noise 0.2, dropout 0.1",
    "QDA": "QDA, reg_param 0.1",
    "XGBoost + LDA feats": "XGBoost on GPU + LDA features",
    "Random forest + LDA feats": "Random forest + LDA features + noise aug.",
    "CatBoost + LDA feats": "CatBoost on GPU + LDA features",
    "FT-Transformer": "FT-Transformer (small: d=32, 2 layers)",
    "Gaussian Naive Bayes": "Gaussian Naive Bayes (raw features)",
    "Decision tree (depth 6)": "Decision tree - depth 6",
}
best = table(list(FAMILY_BEST.values()))
best.index = list(FAMILY_BEST)
best = best.sort_values("test-like")
fig, ax = plt.subplots(figsize=(8, 4.6))
base_acc = best.loc["Logistic regression (final)", "test-like"]
ax.axvline(base_acc, color=AXIS, lw=0.8)
for i, (n, v) in enumerate(best["test-like"].items()):
    win = n.startswith("Logistic")
    ax.plot([v], [i], "o", ms=7, color="#2a78d6" if win else "#b9b7af", mec="white", mew=1.4)
    if win or i == 0:
        ax.annotate(f"{v:.2f}%", (v, i), xytext=(-8 if win else 8, 0), textcoords="offset points",
                    ha="right" if win else "left", va="center", color=INK2)
ax.set_yticks(range(len(best)), best.index)
ax.set_xlabel("accuracy under test-like measurement noise (%), 15-fold paired CV")
ax.grid(axis="y", visible=False)
ax.set_title("Best configuration of each model family")
plt.show()
best.sort_values("test-like", ascending=False)

Every family lands below logistic regression, in rough order of flexibility.

## 4. Ensembles, stacking and threshold tuning

* **Probability averages** of logistic regression with each family's best model.
* **Stacking** – a meta logistic regression learns how to combine 12 models; it is cross-fitted
  (trained on 4 folds' out-of-fold predictions, applied to the 5th).
* **Threshold tuning** – per-class offsets on the log-probabilities, tuned for accuracy on 4 folds
  and applied to the 5th.

In [ ]:
run("Spline GAM on 5 test-noise copies", lambda Xt, yt: make_pipeline(
    SplineTransformer(n_knots=5, degree=3, extrapolation="linear"), StandardScaler(),
    LogisticRegression(C=0.05, max_iter=20000)).fit(*alternating_noise_copies(Xt, yt, 5)))
run("Yeo-Johnson + logistic regression", lambda Xt, yt: make_pipeline(
    PowerTransformer(), LogisticRegression(C=0.1, max_iter=20000)).fit(*jitter(Xt, yt)))
run("Polynomial (degree-2) logistic regression", lambda Xt, yt: make_pipeline(
    PolynomialFeatures(2, include_bias=False), StandardScaler(),
    LogisticRegression(C=0.01, max_iter=20000)).fit(*jitter(Xt, yt)))

ENSEMBLES = {
    "avg: LogReg + spline GAM (5 noisy copies)": [BASE, "Spline GAM on 5 test-noise copies"],
    "avg: LogReg + polynomial LogReg": [BASE, "Polynomial (degree-2) logistic regression"],
    "avg: LogReg + LDA + Naive Bayes": [BASE, "LDA, covariance shrinkage 0.1", "Naive Bayes + LDA(2) projection + aug."],
    "avg: LogReg + SVM": [BASE, "SVM, RBF kernel, C = 1"],
    "avg: LogReg + MLP": [BASE, "MLP 64x2, input noise 0.2, dropout 0.1"],
    "avg: LogReg + random forest": [BASE, "Random forest + LDA features + noise aug."],
    "avg: LogReg + XGBoost": [BASE, "XGBoost on GPU + LDA features"],
    "avg: LogReg + CatBoost": [BASE, "CatBoost on GPU + LDA features"],
}
for name, members in ENSEMBLES.items():
    P[name] = average(members)

TRAIN_V = [("u08", 0), ("u08", 1), ("mix12", 0), ("mix12", 1)]      # the test-like noisy copies

def cross_fit(fit_apply, folds=FOLDS):
    """For each fold: learn on the out-of-fold predictions of the other 4 folds of its repeat."""
    out = [None] * len(folds)
    for r in range(0, len(folds), 5):
        rep = list(range(r, r + 5))
        for i in rep:
            out[i] = fit_apply([j for j in rep if j != i], i)
    return out

STACK_BASES = [BASE, "LDA, covariance shrinkage 0.1", "Naive Bayes + LDA(2) projection + aug.",
               "SVM, RBF kernel, C = 1", "MLP 64x2, input noise 0.2, dropout 0.1",
               "Random forest + LDA features + noise aug.", "XGBoost on GPU + LDA features",
               "CatBoost on GPU + LDA features", "Spline GAM + logistic regression", "QDA, reg_param 0.1",
               "FT-Transformer (small: d=32, 2 layers)", "Yeo-Johnson + logistic regression"]

def stack_fold(others, i):
    feats = lambda j, v, k: np.hstack([np.log(P[b][j][v][k] + 1e-12) for b in STACK_BASES])
    Xm = np.vstack([feats(j, v, k) for j in others for v, k in TRAIN_V])
    tm = np.concatenate([y[FOLDS[j]["va"]] for j in others for _ in TRAIN_V])
    meta = LogisticRegression(C=0.1, max_iter=5000).fit(Xm, tm)
    return {v: [meta.predict_proba(feats(i, v, k)) for k in range(len(ps))] for v, ps in P[BASE][i].items()}

OFFSETS = np.round(np.arange(-0.40, 0.401, 0.02), 2)

def bias_fold(others, i):
    L = np.vstack([np.log(P[BASE][j][v][k] + 1e-12) for j in others for v, k in TRAIN_V])
    t = np.concatenate([y[FOLDS[j]["va"]] for j in others for _ in TRAIN_V])
    best, best_acc = (0.0, 0.0), -1
    for d1 in OFFSETS:
        for d2 in OFFSETS:
            acc = ((L + [0.0, d1, d2]).argmax(1) == t).mean()
            if acc > best_acc + 1e-12 or (abs(acc - best_acc) < 1e-12 and abs(d1) + abs(d2) < abs(best[0]) + abs(best[1])):
                best, best_acc = (d1, d2), acc
    shift = lambda p: (lambda q: q / q.sum(1, keepdims=True))(np.exp(np.log(p + 1e-12) + [0.0, *best]))
    return {v: [shift(p) for p in ps] for v, ps in P[BASE][i].items()}

P["Stacking: meta-model over 12 models"] = cross_fit(stack_fold)
P["Per-class threshold tuning"] = cross_fit(bias_fold)
table(list(ENSEMBLES) + ["Stacking: meta-model over 12 models", "Per-class threshold tuning"])

Averaging never helps: the partners make the same errors as logistic regression plus some of
their own. Stacking and threshold tuning do not help either.

## 5. Robustness to measurement noise

Accuracy as increasing noise (re-clipped and re-rounded) is added to the validation rows of the
first CV repeat.

In [ ]:
LEVELS = (0.0, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40)
CURVE_MODELS = {
    "Logistic regression": lambda Xt, yt: logreg(0.1).fit(*jitter(Xt, yt)),
    "Naive Bayes + LDA(2)": lambda Xt, yt: make_pipeline(StandardScaler(), LDA(n_components=2), GaussianNB()).fit(*jitter(Xt, yt)),
    "QDA (reg 0.1)": lambda Xt, yt: make_pipeline(StandardScaler(), QDA(reg_param=0.1)).fit(Xt, yt),
    "Random forest": lambda Xt, yt: RandomForestClassifier(500, n_jobs=-1, random_state=0).fit(Xt, yt),
    "Decision tree (depth 6)": lambda Xt, yt: tree(max_depth=6, min_samples_leaf=20).fit(Xt, yt),
    "Gaussian Naive Bayes": lambda Xt, yt: GaussianNB().fit(Xt, yt),
}
curves = {}
for name, fit in CURVE_MODELS.items():
    accs = {s: [] for s in LEVELS}
    for f in FOLDS[:5]:
        m = fit(X[f["tr"]], y[f["tr"]])
        sd = X[f["tr"]].std(0, ddof=1)
        for s in LEVELS:
            Zv = test_noise(X[f["va"]], s, 1.0, sd, np.random.default_rng([42, 7, int(s * 100)]))
            accs[s].append((proba(m, Zv).argmax(1) == y[f["va"]]).mean() * 100)
    curves[name] = {s: np.mean(v) for s, v in accs.items()}
curves = pd.DataFrame(curves).T
cols = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
fig, ax = plt.subplots(figsize=(8, 4))
for (name, row), c in zip(curves.iterrows(), cols):
    ax.plot(LEVELS, row.values, color=c, lw=1.8, label=name)
ax.axvspan(0.05, 0.12, color=GRID, alpha=0.6, lw=0, zorder=0)
ax.text(0.085, curves.values.min() + 0.2, "measured\ntest noise", ha="center", fontsize=8, color=MUTED)
ax.set_xlabel("measurement noise added to validation rows (x feature sd)")
ax.set_ylabel("accuracy (%)")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
ax.set_title("Accuracy as measurement noise grows")
plt.show()
curves.round(2)

Logistic regression and Naive Bayes + LDA stay on top at every realistic noise level; the
decision tree collapses fastest, and Gaussian Naive Bayes is flat but low (its error comes from
the wrong independence assumption, not from noise).

## 6. Fresh-fold confirmation

The models closest to the final one are re-run on the 15 confirmation folds. A challenger
would replace logistic regression only if it still won there by more than two standard errors.

In [ ]:
PC = {}
CONFIRM_MODELS = {
    BASE: lambda Xt, yt: logreg(0.1).fit(*jitter(Xt, yt)),
    "Naive Bayes + LDA(2) projection + aug.": lambda Xt, yt: make_pipeline(
        StandardScaler(), LDA(n_components=2), GaussianNB()).fit(*jitter(Xt, yt)),
    "LDA, covariance shrinkage 0.1": lambda Xt, yt: make_pipeline(
        StandardScaler(), LDA(solver="lsqr", shrinkage=0.1)).fit(Xt, yt),
    "Polynomial (degree-2) logistic regression": lambda Xt, yt: make_pipeline(
        PolynomialFeatures(2, include_bias=False), StandardScaler(),
        LogisticRegression(C=0.01, max_iter=20000)).fit(*jitter(Xt, yt)),
    "Spline GAM on 5 test-noise copies": lambda Xt, yt: make_pipeline(
        SplineTransformer(n_knots=5, degree=3, extrapolation="linear"), StandardScaler(),
        LogisticRegression(C=0.05, max_iter=20000)).fit(*alternating_noise_copies(Xt, yt, 5)),
}
for name, fit in CONFIRM_MODELS.items():
    run(name, fit, folds=CONFIRM, store=PC)
PC["avg: LogReg + polynomial LogReg"] = average([BASE, "Polynomial (degree-2) logistic regression"], PC)
PC["avg: LogReg + spline GAM (5 noisy copies)"] = average([BASE, "Spline GAM on 5 test-noise copies"], PC)
table(list(PC), folds=CONFIRM, store=PC)

Nothing beats the final model by more than two standard errors on fresh folds, so the noise-
augmented logistic regression stays.

## 7. Final model and submission file

**Logistic regression** (multinomial, L2 penalty C = 0.1, standardised features) trained on the
7,500 de-duplicated vehicles plus four noise-jittered copies (37,500 rows).

In [ ]:
pred, true = [], []
for i, f in enumerate(FOLDS[:5]):                 # out-of-fold, test-like noise, every row once
    for v in ("u08", "mix12"):
        pred.append(P[BASE][i][v][0].argmax(1))
        true.append(y[f["va"]])
pred, true = np.concatenate(pred), np.concatenate(true)
names = [CLASS[c][0] for c in range(3)]
cm = pd.DataFrame((confusion_matrix(true, pred) / 2).round().astype(int), index=names, columns=names)
prec, rec, f1, _ = precision_recall_fscore_support(true, pred)
cm["precision"], cm["recall"], cm["F1"] = prec.round(3), rec.round(3), f1.round(3)
cm.index.name = "true \\ predicted"
cm

In [ ]:
final = logreg(0.1).fit(*jitter_copies(X, y))
test_pred = final.predict(X_test)
submission = pd.DataFrame({ID: test_ids, TARGET: test_pred.astype(int)})
submission.to_csv("submission_final_logreg.csv", index=False)
sample = pd.read_csv("data/vehicle_sample_submission.csv")
assert submission.shape == sample.shape and (submission[ID] == sample[ID]).all()
print("wrote submission_final_logreg.csv", submission.shape)
print("predicted class mix:", {CLASS[c][0]: f"{(test_pred == c).mean():.1%}" for c in range(3)})
submission.head()

Almost all errors are Sedan ↔ SUV and Sedan ↔ Sports Car; SUVs and sports cars are almost never
confused. The errors sit near the decision boundary – genuine overlap between the classes, which
is why no model gets past about 89.3%. This model scored about **0.90 on the public leaderboard**.

## 8. Experiments with 20% of the training data

Each model is trained on a stratified 20% of every training fold of the first CV repeat and
scored on exactly the same noisy validation rows as with 100% of the data.

In [ ]:
rng = np.random.default_rng(0)
SUB = {}
for f in FOLDS[:5]:
    tr = f["tr"]
    SUB[f["key"]] = np.sort(np.concatenate([rng.choice(tr[y[tr] == c], int(round(0.2 * np.sum(y[tr] == c))),
                                                       replace=False) for c in range(3)]))

def test_like(m, f):
    return np.mean([(proba(m, Z).argmax(1) == y[f["va"]]).mean() for v in ("u08", "mix12")
                    for Z in f["variants"][v]]) * 100

SMALL_MODELS = {
    "Logistic regression": lambda Xt, yt: logreg(0.1).fit(*jitter(Xt, yt)),
    "Decision tree (depth 6)": lambda Xt, yt: tree(max_depth=6, min_samples_leaf=20).fit(Xt, yt),
    "Gaussian Naive Bayes": lambda Xt, yt: GaussianNB().fit(Xt, yt),
    "Naive Bayes + LDA(2)": lambda Xt, yt: make_pipeline(StandardScaler(), LDA(n_components=2), GaussianNB()).fit(*jitter(Xt, yt)),
    "QDA (reg 0.1)": lambda Xt, yt: make_pipeline(StandardScaler(), QDA(reg_param=0.1)).fit(Xt, yt),
    "SVM (RBF, C=1)": lambda Xt, yt: make_pipeline(StandardScaler(), SVC(C=1.0, random_state=0)).fit(Xt, yt),
    "Random forest": lambda Xt, yt: RandomForestClassifier(500, n_jobs=-1, random_state=0).fit(Xt, yt),
    "XGBoost + LDA features": lambda Xt, yt: WithLDA(xgb(2, 600, 0.03)).fit(Xt, yt),
}
exp20 = {}
for name, fit in SMALL_MODELS.items():
    exp20[name] = {"100% data": np.mean([test_like(fit(X[f["tr"]], y[f["tr"]]), f) for f in FOLDS[:5]]),
                   "20% data": np.mean([test_like(fit(X[SUB[f["key"]]], y[SUB[f["key"]]]), f) for f in FOLDS[:5]])}
eval_sets = [np.vstack([Z for v in ("u08", "mix12") for Z in f["variants"][v]]) for f in FOLDS[:5]]
mlp_s = [dict(noise=0.2, share=1.0, wd=0.01, drop=0.1)]
for label, idx_of, epochs in (("100% data", lambda f: f["tr"], 80), ("20% data", lambda f: SUB[f["key"]], 400)):
    probs = train_mlps([(X[idx_of(f)], y[idx_of(f)]) for f in FOLDS[:5]], eval_sets, 64, 2, mlp_s, epochs=epochs)
    exp20.setdefault("Neural network (MLP 64x2)", {})[label] = np.mean(
        [(probs[(0, i)].argmax(1) == np.tile(y[f["va"]], 4)).mean() * 100 for i, f in enumerate(FOLDS[:5])])
exp20 = pd.DataFrame(exp20).T
exp20["change"] = exp20["20% data"] - exp20["100% data"]
exp20 = exp20.sort_values("100% data", ascending=False)

fig, ax = plt.subplots(figsize=(8, 3.8))
rows = exp20.sort_values("100% data")
for i, (n, r) in enumerate(rows.iterrows()):
    ax.plot([r["20% data"], r["100% data"]], [i, i], color=AXIS, lw=1.6, zorder=1)
    ax.plot(r["100% data"], i, "o", ms=7, color="#1c5cab", mec="white", mew=1.2, label="100% of training rows" if i == 0 else None)
    ax.plot(r["20% data"], i, "o", ms=7, color="#86b6ef", mec="white", mew=1.2, label="20% of training rows" if i == 0 else None)
ax.set_yticks(range(len(rows)), rows.index)
ax.set_xlabel("accuracy under test-like noise (%), same validation rows")
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right")
ax.set_title("Training on 20% of the data")
plt.show()
print(f"training rows per fold: {len(FOLDS[0]['tr']):,} (100%) vs {len(SUB[FOLDS[0]['key']]):,} (20%)")
exp20.round(2)

Models with few parameters barely notice (Naive Bayes + LDA, Gaussian Naive Bayes); logistic
regression and the SVM lose about 0.6 points; flexible models need the data most – the MLP and
the decision tree lose well over a point.

## 9. Unsupervised learning

Clustering on the standardised features, without labels: do the three vehicle types form
natural groups? *Matched accuracy* = accuracy after optimally matching clusters to classes.

In [ ]:
Zs = StandardScaler().fit_transform(X)
eps = float(np.quantile(np.sort(NearestNeighbors(n_neighbors=11).fit(Zs).kneighbors(Zs)[0][:, 10]), 0.90))

def matched(labels):
    ks = [k for k in np.unique(labels) if k != -1]
    M = np.array([[np.sum((labels == k) & (y == c)) for c in range(3)] for k in ks])
    r, c = linear_sum_assignment(-M)
    return M[r, c].sum() / len(y) * 100, {int(ks[i]): int(c[j]) for j, i in enumerate(r)}

ALGOS = {
    "K-Means (k=3)": lambda: KMeans(3, n_init=10, random_state=0).fit_predict(Zs),
    "Gaussian mixture (k=3)": lambda: GaussianMixture(3, covariance_type="full", n_init=3, random_state=0).fit(Zs).predict(Zs),
    "Agglomerative (Ward, k=3)": lambda: AgglomerativeClustering(3, linkage="ward").fit_predict(Zs),
    f"DBSCAN (eps = {eps:.2f})": lambda: DBSCAN(eps=eps, min_samples=10).fit_predict(Zs),
}
rows, labels_km = [], None
for name, fit in ALGOS.items():
    lab = fit()
    n_cl = len([k for k in np.unique(lab) if k != -1])
    acc, mapping = matched(lab)
    if name.startswith("K-Means"):
        labels_km, map_km = lab, mapping
    rows.append({"algorithm": name, "clusters": n_cl, "noise points": int((lab == -1).sum()),
                 "silhouette": silhouette_score(Zs[lab != -1], lab[lab != -1]) if n_cl > 1 else np.nan,
                 "ARI": adjusted_rand_score(y, lab), "homogeneity": homogeneity_score(y, lab),
                 "matched accuracy %": acc})
unsup = pd.DataFrame(rows).set_index("algorithm").round(3)
display(unsup)

T = PCA(2).fit_transform(Zs)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, lab, title in ((axes[0], y, "True classes"),
                       (axes[1], np.array([map_km.get(int(k), -1) for k in labels_km]),
                        "K-Means clusters (coloured by matched class)")):
    for c, (name, col) in CLASS.items():
        ax.scatter(T[lab == c, 0], T[lab == c, 1], s=2, color=col, alpha=0.45, linewidths=0, label=name)
    ax.set_title(title)
    ax.set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].legend(markerscale=5, loc="lower left")
plt.show()

K-Means recovers the vehicle types surprisingly well (about 81% after matching), because the
classes are roughly spherical clouds of similar size. DBSCAN finds a single dense cluster – the
classes touch, so there are no low-density gaps to cut along. Even the best clustering is about
eight points below supervised learning: useful for exploration, not a replacement for it.

## 10. Key insights

1. **The boundary is linear, so logistic regression wins.** Every more flexible family scored
   lower, in rough order of flexibility.
2. **Naive Bayes needs the right coordinate system** – on raw correlated features it is almost
   five points behind; on the LDA projection it ties logistic regression.
3. **Removing duplicates is essential** for honest validation (`02_Preprocessing`).
4. **Measuring the test noise beats guessing it** – every model was scored under realistic noise.
5. **The regularised linear model is both the most accurate and among the most noise-robust.**
6. **Fresh-fold confirmation prevents false improvements** (the Yeo-Johnson lesson in `02`).
7. **About 89.3% is the ceiling of this dataset**: the remaining error is genuine overlap between
   sedans and the other two types.